# MelBERT replication on Google Colab (GPU)

This runs the **full** VUA-18 replication with the paper's hyperparameters, plus the zero-shot transfer sets and the new-domain test sets.

1. `Runtime → Change runtime type → GPU` (T4 is enough; L4/A100 are faster).
2. Run the cells in order. **Everything is copied to your Google Drive** (`MyDrive/melbert_results`) so a disconnect can't lose your checkpoint or results.

Rough timing on a T4: ~1–1.5 h per epoch on VUA-18 (3 epochs) plus test-set evaluation after each epoch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/<YOUR-GITHUB-USERNAME>/melbert-reproduction.git"   # <-- edit
!git clone -q $REPO_URL /content/melbert-reproduction
%cd /content/melbert-reproduction
!pip -q install -r requirements.txt gdown
!bash setup/setup_melbert.sh

## 2. Save everything to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/melbert_results'
!mkdir -p $DRIVE_OUT

## 3. Download the official pre-processed datasets
These are the authors' pre-processed VUA-18 / VUA-20 / VUA-Verb / MOH-X / TroFi files (linked from the official README).

In [ ]:
!gdown -q --fuzzy "https://drive.google.com/file/d/18hemekvUuOw-qkQhWv4F6qFnis6lkaog/view" -O /content/melbert_data.zip
!unzip -q -o /content/melbert_data.zip -d /content/melbert_data_raw
!find /content/melbert_data_raw -maxdepth 3 -type d | head -40

In [ ]:
# Move the dataset folders into data/ (handles an extra top-level folder in the zip)
import os, shutil, glob
wanted = ["VUA18", "VUA20", "VUAverb", "MOH-X", "TroFi"]
for name in wanted:
    hits = [p for p in glob.glob(f"/content/melbert_data_raw/**/{name}", recursive=True) if os.path.isdir(p)]
    if hits and not os.path.exists(f"data/{name}"):
        shutil.copytree(hits[0], f"data/{name}")
    print(name, "->", "ok" if os.path.exists(f"data/{name}") else "NOT FOUND (check the find output above)")
!wc -l data/VUA18/*.tsv
!python src/prepare_transfer_sets.py --data_root data

## 4. (Optional, fast) Score the authors' released checkpoint
This separates "does the code + data reproduce the paper?" from "does training reproduce it?". It takes a few minutes.

In [ ]:
!gdown -q --fuzzy "https://drive.google.com/file/d/1RKyP-_kje6xWlNuXeoSIHZnYxUyXnupZ/view" -O /content/official_ckpt.zip
!mkdir -p checkpoints/official && (unzip -q -o /content/official_ckpt.zip -d checkpoints/official || mv /content/official_ckpt.zip checkpoints/official/pytorch_model.bin)
!find checkpoints/official | head
!bash scripts/run_predictions.sh checkpoints/official official
!cp -r results $DRIVE_OUT/

## 5. Full replication training (paper settings, seed 42)

In [ ]:
!bash scripts/run_replication.sh 42
!cp -r external/MelBERT/saves $DRIVE_OUT/
!cp -r results $DRIVE_OUT/

### Optional: extra seeds
The paper averages 5 seeds. If you have GPU time left, run more seeds (each is a full training run) and report the mean ± std.

In [ ]:
# for s in [1, 2]:
#     !bash scripts/run_replication.sh {s}
#     !cp -r external/MelBERT/saves $DRIVE_OUT/ && cp -r results $DRIVE_OUT/

## 6. Predictions with your trained checkpoint on every test set

In [ ]:
import glob, os
ckpt = sorted(glob.glob("external/MelBERT/saves/roberta-base/*/pytorch_model.bin"), key=os.path.getmtime)[-1]
CKPT_DIR = os.path.dirname(ckpt); print(CKPT_DIR)
!bash scripts/run_predictions.sh $CKPT_DIR ours
!cp -r results $DRIVE_OUT/

## 7. Metrics + LaTeX tables

In [ ]:
!python src/evaluate.py --tag ours --tag2 official --llm_labels data/annotation/labels_LLM.csv
!cat results/tables/replication.tex
!cp -r results $DRIVE_OUT/